In [3]:
import sys
import os

# Add the parent directory to the Python path
sys.path.append(os.path.abspath('..'))

### Cell 1: Manual Gradient Descent
    
This cell builds a neural network and trains it using basic, manual Gradient Descent.

In [ ]:

from src.mlp import MLP
    
# 1. Initialize a 2-layer neural network
# We have 2 inputs (e.g., x1, x2).
# We define a hidden layer architecture of [4, 4, 1]:
# - First hidden layer: 4 neurons
# - Second hidden layer: 4 neurons
# - Output layer: 1 neuron (providing a single scalar prediction)
model = MLP(2, [4, 4, 1])
    
# 2. Define the XOR-like dataset
# xs contains 4 training examples, each with 2 features.
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [-1.0, -3.0],
    [-1.0, 1.0],
]
# ys contains the target outputs for each example (1.0 or -1.0).
ys = [1.0, -1.0, -1.0, 1.0]
    
# 3. Training Loop (Gradient Descent)
epochs = 50
learning_rate = 0.05
    
for k in range(epochs):
    # --- A. Forward Pass ---
    # Pass all 4 input examples through the network to get 4 predictions.
    # Because we are using our custom Autograd engine, this builds a computational graph in the background.
    ypred = [model(x) for x in xs]
        
    # --- B. Calculate Loss ---
    # Calculate Mean Squared Error (MSE) loss.
    # This measures how far off our predictions are from the true targets.
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
        
    # --- C. Backward Pass ---
    # Crucial: Reset the gradients of all weights/biases to 0 before computing new ones.
    # If we don't do this, the gradients from the previous loop will accumulate (add up).
    model.zero_grad() 
        
    # Trigger reverse-mode autodiff! This traverses the computational graph backwards,
    # calculating the exact derivative of the `loss` with respect to every weight in the network.
    loss.backward()
        
    # --- D. Update Weights (Gradient Descent) ---
    # Iterate over every parameter (weight and bias) in the model.
    for p in model.parameters():
        # Update the parameter by taking a small step in the opposite direction of the gradient.
        # (gradient points towards the steepest increase in loss, so we subtract it to decrease loss).
        p.data -= learning_rate * p.grad
            
    print(f"Epoch {k} | Loss: {loss.data:.4f}")

# 4. Verify final predictions
print("\nFinal Predictions:")
for x, target, pred in zip(xs, ys, ypred):
    print(f"Input: {x} | Target: {target} | Pred: {pred.data:.4f}")


Epoch 0 | Loss: 10.5502
Epoch 1 | Loss: 2.3709
Epoch 2 | Loss: 0.7809
Epoch 3 | Loss: 0.1732
Epoch 4 | Loss: 0.0242
Epoch 5 | Loss: 0.0043
Epoch 6 | Loss: 0.0020
Epoch 7 | Loss: 0.0015
Epoch 8 | Loss: 0.0011
Epoch 9 | Loss: 0.0009
Epoch 10 | Loss: 0.0007
Epoch 11 | Loss: 0.0005
Epoch 12 | Loss: 0.0004
Epoch 13 | Loss: 0.0003
Epoch 14 | Loss: 0.0002
Epoch 15 | Loss: 0.0002
Epoch 16 | Loss: 0.0001
Epoch 17 | Loss: 0.0001
Epoch 18 | Loss: 0.0001
Epoch 19 | Loss: 0.0001
Epoch 20 | Loss: 0.0000
Epoch 21 | Loss: 0.0000
Epoch 22 | Loss: 0.0000
Epoch 23 | Loss: 0.0000
Epoch 24 | Loss: 0.0000
Epoch 25 | Loss: 0.0000
Epoch 26 | Loss: 0.0000
Epoch 27 | Loss: 0.0000
Epoch 28 | Loss: 0.0000
Epoch 29 | Loss: 0.0000
Epoch 30 | Loss: 0.0000
Epoch 31 | Loss: 0.0000
Epoch 32 | Loss: 0.0000
Epoch 33 | Loss: 0.0000
Epoch 34 | Loss: 0.0000
Epoch 35 | Loss: 0.0000
Epoch 36 | Loss: 0.0000
Epoch 37 | Loss: 0.0000
Epoch 38 | Loss: 0.0000
Epoch 39 | Loss: 0.0000
Epoch 40 | Loss: 0.0000
Epoch 41 | Loss: 0.0000
E

### Cell 2: Training with Adam Optimizer

This cell does the exact same task, but instead of updating weights manually, it delegates the update logic to an Adam optimizer (which is an advanced, faster version of Gradient Descent).


In [ ]:
from src.mlp import MLP
from src.optimizers import Adam

# 1. Initialize network and optimizer
model = MLP(2, [4, 4, 1])

# Pass the model's parameters to the Adam optimizer.
# Adam generally converges faster and uses a smaller default learning rate than pure SGD.
# It maintains "momentum" (running averages of gradients) internally for smoother updates.
optimizer = Adam(model.parameters(), lr=0.05) 

# Dataset
xs = [[2.0, 3.0], [3.0, -1.0], [-1.0, -3.0], [-1.0, 1.0]]
ys = [1.0, -1.0, -1.0, 1.0]

# 2. Training Loop
for epoch in range(50):
    # --- A. Forward Pass ---
    # Generate predictions and compute the total loss for the batch.
    ypred = [model(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
        
    # --- B. Backward Pass and Optimization ---
    # 1. Flush previous gradients (the optimizer delegates this to the model's parameters).
    optimizer.zero_grad()  
        
    # 2. Compute the new gradients for the current loss via backpropagation.
    loss.backward()        
        
    # 3. Apply the Adam update equations. 
    # Instead of doing `p.data -= lr * p.grad` manually, the optimizer handles the complex math.
    optimizer.step()       
            
    print(f"Epoch {epoch} | Loss: {loss.data:.4f}")


Epoch 0 | Loss: 6.3277
Epoch 1 | Loss: 4.0886
Epoch 2 | Loss: 2.5687
Epoch 3 | Loss: 1.6227
Epoch 4 | Loss: 1.0174
Epoch 5 | Loss: 0.5849
Epoch 6 | Loss: 0.2912
Epoch 7 | Loss: 0.1325
Epoch 8 | Loss: 0.0774
Epoch 9 | Loss: 0.0911
Epoch 10 | Loss: 0.1418
Epoch 11 | Loss: 0.1964
Epoch 12 | Loss: 0.2365
Epoch 13 | Loss: 0.2548
Epoch 14 | Loss: 0.2449
Epoch 15 | Loss: 0.2068
Epoch 16 | Loss: 0.1526
Epoch 17 | Loss: 0.0975
Epoch 18 | Loss: 0.0520
Epoch 19 | Loss: 0.0233
Epoch 20 | Loss: 0.0158
Epoch 21 | Loss: 0.0281
Epoch 22 | Loss: 0.0497
Epoch 23 | Loss: 0.0652
Epoch 24 | Loss: 0.0648
Epoch 25 | Loss: 0.0496
Epoch 26 | Loss: 0.0289
Epoch 27 | Loss: 0.0119
Epoch 28 | Loss: 0.0034
Epoch 29 | Loss: 0.0032
Epoch 30 | Loss: 0.0076
Epoch 31 | Loss: 0.0131
Epoch 32 | Loss: 0.0176
Epoch 33 | Loss: 0.0204
Epoch 34 | Loss: 0.0209
Epoch 35 | Loss: 0.0187
Epoch 36 | Loss: 0.0143
Epoch 37 | Loss: 0.0095
Epoch 38 | Loss: 0.0059
Epoch 39 | Loss: 0.0038
Epoch 40 | Loss: 0.0031
Epoch 41 | Loss: 0.0033
Ep